# 🎭 Pipeline Completo — Ataque Deepfake com Câmera Virtual

**Projeto:** Avaliação de Vulnerabilidade de Sistemas Biométricos Faciais contra Deepfakes

**Ferramentas:** FaceFusion (inswapper_128) + AdaFace IR-50 + YOLOv8 + v4l2loopback

---

## Arquitetura do Ataque

```
┌─────────────┐    ┌─────────────┐    ┌──────────────────┐    ┌─────────────────┐    ┌──────────────┐
│ Foto do      │    │ Vídeo da    │    │   FaceFusion     │    │ Câmera Virtual  │    │ Zoom / Meet  │
│ Atacante     │───▶│ Vítima      │───▶│ (Face Swap)      │───▶│ (v4l2loopback)  │───▶│ Site Banco   │
│ (.jpeg)      │    │ (.mp4)      │    │                  │    │ /dev/video7     │    │              │
└─────────────┘    └─────────────┘    └──────────────────┘    └─────────────────┘    └──────────────┘
```

---
## Etapa 1 — Preparar os Arquivos de Entrada

Coloque a **FOTO** (rosto do invasor) e o **VÍDEO** (corpo/cenário alvo) na pasta `data/`.

### ⚠️ Execute no terminal Ubuntu (FORA do Docker):

In [ ]:
# ============================================================
# 📌 CONFIGURAÇÃO: Altere os nomes dos seus arquivos aqui!
# ============================================================

NOME_DA_FOTO = "WhatsApp Image 2026-04-20 at 20.33.31.jpeg"   # ← Troque pelo nome da SUA FOTO
NOME_DO_VIDEO = "WhatsApp Video 2026-04-20 at 20.33.33.mp4"   # ← Troque pelo nome do SEU VÍDEO

# ============================================================
# NÃO ALTERE NADA ABAIXO
# ============================================================
import os
print(f"📸 Foto do atacante: {NOME_DA_FOTO}")
print(f"🎬 Vídeo alvo:       {NOME_DO_VIDEO}")

In [ ]:
# Copia os arquivos de Downloads para dentro do projeto
# (Execute isso FORA do Docker, no terminal Ubuntu)

import shutil
import os

home = os.path.expanduser("~")
downloads = os.path.join(home, "Downloads")
data_dir = os.path.join(home, "deepfaketeste", "data")

src_foto = os.path.join(downloads, NOME_DA_FOTO)
src_video = os.path.join(downloads, NOME_DO_VIDEO)
dst_foto = os.path.join(data_dir, "ataque_foto.jpeg")
dst_video = os.path.join(data_dir, "ataque_video.mp4")

# Verifica se os arquivos existem
assert os.path.exists(src_foto), f"❌ Foto não encontrada: {src_foto}"
assert os.path.exists(src_video), f"❌ Vídeo não encontrado: {src_video}"

shutil.copy2(src_foto, dst_foto)
shutil.copy2(src_video, dst_video)

print(f"✅ Foto copiada para: {dst_foto} ({os.path.getsize(dst_foto) / 1024:.0f} KB)")
print(f"✅ Vídeo copiado para: {dst_video} ({os.path.getsize(dst_video) / 1024 / 1024:.1f} MB)")

---
## Etapa 2 — Fabricar o Deepfake com FaceFusion

O FaceFusion lê a foto do invasor, abre o vídeo quadro a quadro, e cola o rosto por cima.

### ⚠️ Execute dentro do Docker (`sudo docker exec -it ia_gpu bash`)

In [ ]:
# ============================================================
# FABRICAÇÃO DO DEEPFAKE
# Cole este comando diretamente no terminal DOCKER:
# ============================================================

comando_facefusion = """
cd /app/facefusion && python facefusion.py headless-run \\
  --processors face_swapper \\
  --face-swapper-model inswapper_128 \\
  -s /app/data/ataque_foto.jpeg \\
  -t /app/data/ataque_video.mp4 \\
  -o /app/data/deepfake_final.mp4 \\
  --execution-providers cuda
"""

print("📋 Copie e cole este comando no terminal DOCKER:")
print("=" * 60)
print(comando_facefusion)
print("=" * 60)
print("\n⏳ Aguarde até o terminal voltar ao prompt (1-3 min para vídeos curtos)")

In [ ]:
# Verificar se o deepfake foi gerado com sucesso
import os

deepfake_path = os.path.expanduser("~/deepfaketeste/data/deepfake_final.mp4")

if os.path.exists(deepfake_path) and os.path.getsize(deepfake_path) > 0:
    size_mb = os.path.getsize(deepfake_path) / 1024 / 1024
    print(f"✅ Deepfake gerado com sucesso!")
    print(f"📁 Arquivo: {deepfake_path}")
    print(f"📏 Tamanho: {size_mb:.1f} MB")
else:
    print("❌ Deepfake NÃO encontrado. Execute o FaceFusion no Docker primeiro!")

---
## Etapa 3 — Visualizar Original vs Deepfake

Comparação visual lado a lado entre o vídeo original e o deepfake gerado.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

data_dir = os.path.expanduser("~/deepfaketeste/data")

# Carrega a foto do atacante
foto = cv2.imread(os.path.join(data_dir, "ataque_foto.jpeg"))
foto_rgb = cv2.cvtColor(foto, cv2.COLOR_BGR2RGB) if foto is not None else None

# Pega o primeiro frame do vídeo original
cap_orig = cv2.VideoCapture(os.path.join(data_dir, "ataque_video.mp4"))
ret1, frame_orig = cap_orig.read()
cap_orig.release()
frame_orig_rgb = cv2.cvtColor(frame_orig, cv2.COLOR_BGR2RGB) if ret1 else None

# Pega o primeiro frame do deepfake
cap_fake = cv2.VideoCapture(os.path.join(data_dir, "deepfake_final.mp4"))
ret2, frame_fake = cap_fake.read()
cap_fake.release()
frame_fake_rgb = cv2.cvtColor(frame_fake, cv2.COLOR_BGR2RGB) if ret2 else None

# Plota lado a lado
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

if foto_rgb is not None:
    axes[0].imshow(foto_rgb)
axes[0].set_title("📸 Foto do Atacante\n(rosto que será colado)", fontsize=13, fontweight='bold')
axes[0].axis('off')

if frame_orig_rgb is not None:
    axes[1].imshow(frame_orig_rgb)
axes[1].set_title("🎬 Vídeo Original\n(antes do ataque)", fontsize=13, fontweight='bold')
axes[1].axis('off')

if frame_fake_rgb is not None:
    axes[2].imshow(frame_fake_rgb)
axes[2].set_title("🎭 Vídeo Deepfake\n(após face swap)", fontsize=13, fontweight='bold', color='red')
axes[2].axis('off')

plt.suptitle("Comparação: Original vs Deepfake", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.expanduser("~/deepfaketeste/app/results/comparacao_visual.png"), dpi=150)
plt.show()
print("📊 Imagem salva em: app/results/comparacao_visual.png")

---
## Etapa 4 — Ativar a Câmera Virtual + Injetar o Deepfake

### ⚠️ Execute FORA do Docker (terminal Ubuntu normal)

In [ ]:
# ============================================================
# PASSO 1: Ativar a câmera virtual (só precisa 1x por boot)
# Cole no terminal Ubuntu:
# ============================================================

cmd_camera = "sudo modprobe v4l2loopback devices=1 video_nr=7 card_label='Deepfake Virtual Cam' exclusive_caps=1"

print("📋 PASSO 1 — Ativar câmera virtual:")
print("=" * 60)
print(cmd_camera)
print("=" * 60)

# ============================================================
# PASSO 2: Injetar o deepfake na câmera
# Cole no terminal Ubuntu:
# ============================================================

cmd_inject = """sudo ffmpeg -re -stream_loop -1 \\
  -i ~/deepfaketeste/data/deepfake_final.mp4 \\
  -vf "format=yuv420p" \\
  -f v4l2 /dev/video7"""

print("\n📋 PASSO 2 — Injetar deepfake na câmera:")
print("=" * 60)
print(cmd_inject)
print("=" * 60)
print("\n🔴 O terminal vai ficar 'travado' — isso é normal!")
print("   Para parar, aperte Ctrl+C")

In [ ]:
# ============================================================
# PASSO 3: Testar!
# ============================================================

print("""
🌐 PASSO 3 — Abra um destes sites no navegador:

   1. https://webcamtests.com
   2. https://meet.google.com
   3. https://zoom.us

📹 Nas configurações de câmera, selecione:
   → "Deepfake Virtual Cam"

🎭 Você verá o rosto trocado aparecendo como se fosse
   uma pessoa real na frente do computador!
""")

---
## Etapa 5 — Verificação Biométrica (Prova para o TCC)

Verificamos matematicamente se o deepfake gerado consegue enganar o sistema AdaFace.

### ⚠️ Execute dentro do Docker

In [ ]:
# ============================================================
# VERIFICAÇÃO BIOMÉTRICA - Cole no terminal DOCKER
# ============================================================

script_verificacao = '''
cd /app && python -c "
import cv2, numpy as np
from baseline_lfw import load_adaface, preprocess_face, get_embedding, detect_and_crop_face, DEVICE, ADAFACE_CKPT, YOLO_FACE_PATH
from ultralytics import YOLO

yolo = YOLO(YOLO_FACE_PATH, verbose=False)
model = load_adaface(ADAFACE_CKPT).to(DEVICE)

foto_original = cv2.imread('data/ataque_foto.jpeg')
cap = cv2.VideoCapture('data/deepfake_final.mp4')
ret, frame_fake = cap.read()
cap.release()

face1 = detect_and_crop_face(yolo, foto_original, 112)
face2 = detect_and_crop_face(yolo, frame_fake, 112)

emb1 = get_embedding(model, preprocess_face(face1), DEVICE)
emb2 = get_embedding(model, preprocess_face(face2), DEVICE)

sim = float(np.dot(np.squeeze(emb1), np.squeeze(emb2)))
print(f'\\n=======================================')
print(f'  RESULTADO DA VERIFICACAO BIOMETRICA')
print(f'=======================================')
print(f'Similaridade: {sim:.4f}')
print(f'Threshold 0.45: {chr(128680) + \" ENGANOU\" if sim >= 0.45 else chr(128737) + \" BARRADO\"}')
print(f'Threshold 0.60: {chr(128680) + \" ENGANOU\" if sim >= 0.60 else chr(128737) + \" BARRADO\"}')
print(f'=======================================\\n')
"
'''

print("📋 Cole este comando no terminal DOCKER:")
print("=" * 60)
print(script_verificacao)
print("=" * 60)

---
## 📊 Resultados Consolidados do Projeto

### Tabela comparativa de todos os experimentos realizados

In [ ]:
import pandas as pd
import os

# Resultados coletados dos experimentos
resultados = {
    "Experimento": [
        "Baseline LFW (Fotos)",
        "Baseline YTF (Vídeos)",
        "Ataque LFW (Face Swap em Foto)",
        "Ataque YTF (Face Swap em Vídeo)",
        "Câmera Virtual (Tempo Real)"
    ],
    "Dataset": [
        "LFW (Labeled Faces in the Wild)",
        "YTF (YouTube Faces)",
        "LFW + FaceFusion",
        "YTF + FaceFusion",
        "Vídeo pessoal + FaceFusion"
    ],
    "Tipo": [
        "Foto estática",
        "Vídeo (5 frames)",
        "Foto manipulada",
        "Vídeo manipulado",
        "Stream ao vivo"
    ],
    "Métrica Principal": [
        "AUC = 0.99+",
        "AUC = 0.8210",
        "ASR (τ=0.45) = 93.0%",
        "ASR (τ=0.45) = 20.0%",
        "Verificar manualmente"
    ],
    "Conclusão": [
        "✅ Sistema confiável sem ataques",
        "✅ Sistema funciona razoavelmente bem",
        "🚨 Sistema VULNERÁVEL a deepfakes em foto",
        "🛡️ Sistema mais RESILIENTE em vídeo",
        "🎭 Demonstração de viabilidade do ataque"
    ]
}

df = pd.DataFrame(resultados)
print("\n" + "=" * 100)
print(" 📊 RESULTADOS CONSOLIDADOS DO PROJETO ".center(100, "="))
print("=" * 100)
print(df.to_string(index=False))
print("\n")

# Salva CSV
csv_path = os.path.expanduser("~/deepfaketeste/app/results/resultados_consolidados.csv")
df.to_csv(csv_path, index=False)
print(f"📁 Tabela salva em: {csv_path}")

---
## 🔑 Conclusões para o TCC

| Descoberta | Evidência |
|:-----------|:----------|
| **Fotos são altamente vulneráveis** | ASR de 93% no LFW mostra que deepfakes em imagem estática enganam facilmente sistemas baseados em AdaFace |
| **Vídeos são mais resilientes** | ASR caiu para 20% no YTF — a amostragem temporal (múltiplos frames) dificulta a falsificação consistente |
| **Câmera virtual é viável** | O módulo v4l2loopback permite injetar deepfakes em tempo real em qualquer aplicação que use webcam |
| **Defesa recomendada** | Sistemas biométricos devem usar verificação multi-frame com análise temporal, não apenas snapshots isolados |

---

### 📚 Referências Técnicas

- **AdaFace**: Kim et al., "AdaFace: Quality Adaptive Margin for Face Recognition", CVPR 2022
- **FaceFusion**: https://github.com/facefusion/facefusion
- **YOLOv8**: Ultralytics, 2023
- **LFW**: Huang et al., "Labeled Faces in the Wild", 2007
- **YTF**: Wolf et al., "Face Recognition in Unconstrained Videos", CVPR 2011
- **v4l2loopback**: https://github.com/umlaeute/v4l2loopback

---
## 🔁 Quer Testar com Outra Pessoa?

1. Volte na **Etapa 1** e altere `NOME_DA_FOTO` e `NOME_DO_VIDEO`
2. Execute as células na ordem
3. Repita a fabricação no Docker (Etapa 2)
4. Injete na câmera novamente (Etapa 4)

> **Dica:** Quanto mais parecidas forem as condições de iluminação entre a foto e o vídeo, melhor será o resultado do deepfake!